# IntelliView — Production-Grade SER Training (GPU)

Trains a **2D-CNN** on mel-spectrograms of the RAVDESS dataset with **6x data augmentation**.

**Expected accuracy:** 70-80% (8-class emotion classification)
**Training time:** ~30-45 min on T4 GPU

## Before Running
1. Go to `Runtime` → `Change runtime type` → select **T4 GPU** → Save
2. Run each cell in order (Shift + Enter)
3. The last cell downloads the trained model to your computer

## Cell 1 — Verify GPU & Install Dependencies

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {len(gpus) > 0}')
if gpus:
    print(f'   Device: {gpus[0]}')
else:
    print('⚠️ No GPU — go to Runtime > Change runtime type > T4 GPU')

# Install audio libraries
!pip install librosa soundfile --quiet
print('✅ Dependencies ready')

## Cell 2 — Download RAVDESS Dataset (~250 MB)

In [ ]:
import os
import urllib.request
import zipfile
import glob

DATA_DIR = '/content/ravdess'
os.makedirs(DATA_DIR, exist_ok=True)

existing = glob.glob(os.path.join(DATA_DIR, '**', '*.wav'), recursive=True)
if len(existing) >= 1400:
    print(f'✅ Already downloaded ({len(existing)} files)')
else:
    url = 'https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip?download=1'
    zip_path = '/content/ravdess.zip'
    print('📦 Downloading RAVDESS (~250 MB)...')
    urllib.request.urlretrieve(url, zip_path)
    print('📂 Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    os.remove(zip_path)
    files = glob.glob(os.path.join(DATA_DIR, '**', '*.wav'), recursive=True)
    print(f'✅ Extracted {len(files)} audio files')

## Cell 3 — Feature Extraction + 6x Augmentation

Each original clip becomes 6 training samples:
1. Original
2. Pitch shift +2 semitones
3. Pitch shift -2 semitones
4. Time stretch 0.9x (slower)
5. Time stretch 1.1x (faster)
6. Gaussian noise injection

Features: **128 mel-bands × 130 time frames** (full mel-spectrogram — not averaged)

In [ ]:
import numpy as np
import librosa

SAMPLE_RATE = 22050
DURATION = 3
N_MELS = 128
N_FRAMES = 130  # mel-spec time dimension for 3s @ 22050 Hz, hop=512

EMOTION_MAP = {
    '01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
    '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised'
}

def get_mel(y, sr):
    """Extract log-mel-spectrogram and pad/crop to fixed size."""
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    # Pad or truncate to N_FRAMES
    if mel_db.shape[1] < N_FRAMES:
        pad = N_FRAMES - mel_db.shape[1]
        mel_db = np.pad(mel_db, ((0,0),(0,pad)), mode='constant', constant_values=mel_db.min())
    else:
        mel_db = mel_db[:, :N_FRAMES]
    return mel_db

def augment_and_extract(file_path):
    """Load one audio file, produce 6 augmented variants as mel-spectrograms."""
    try:
        y, sr = librosa.load(file_path, duration=DURATION, sr=SAMPLE_RATE)
        if len(y) < SAMPLE_RATE * DURATION:
            y = np.pad(y, (0, SAMPLE_RATE * DURATION - len(y)), mode='constant')

        variants = []
        variants.append(y)                                           # 1. Original
        variants.append(librosa.effects.pitch_shift(y, sr=sr, n_steps=2))   # 2. Higher pitch
        variants.append(librosa.effects.pitch_shift(y, sr=sr, n_steps=-2))  # 3. Lower pitch
        variants.append(librosa.effects.time_stretch(y, rate=0.9))   # 4. Slower
        variants.append(librosa.effects.time_stretch(y, rate=1.1))   # 5. Faster
        variants.append(y + 0.005 * np.random.randn(len(y)))         # 6. Noise

        return [get_mel(v, sr) for v in variants]
    except Exception as e:
        return None

print('🔬 Extracting features with 6x augmentation...')
print('   This takes ~5-10 min — grab a coffee ☕\n')

audio_files = glob.glob(os.path.join(DATA_DIR, '**', '*.wav'), recursive=True)

features = []
labels = []
total = len(audio_files)

for i, f in enumerate(audio_files):
    parts = os.path.basename(f).split('-')
    if len(parts) < 3:
        continue
    emo = EMOTION_MAP.get(parts[2])
    if emo is None:
        continue

    mels = augment_and_extract(f)
    if mels is None:
        continue

    for mel in mels:
        features.append(mel)
        labels.append(emo)

    if (i+1) % 100 == 0 or i == total-1:
        print(f'   Processed {i+1}/{total} original files → {len(features)} augmented samples')

X = np.array(features)
y = np.array(labels)
print(f'\n✅ Feature matrix: {X.shape}   (samples, mel_bands, time_frames)')
print(f'✅ Labels:         {y.shape}')
print(f'\nEmotion distribution:')
import collections
for emo, count in sorted(collections.Counter(y).items()):
    print(f'   {emo:12s}: {count}')

## Cell 4 — Build & Train 2D-CNN on GPU

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization, Dropout,
    Flatten, Dense, GlobalAveragePooling2D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y)
y_cat = to_categorical(y_enc)
num_classes = y_cat.shape[1]
print(f'Classes: {list(le.classes_)}')

# Normalize per-sample (z-score on each mel-spectrogram)
X_norm = (X - X.mean(axis=(1,2), keepdims=True)) / (X.std(axis=(1,2), keepdims=True) + 1e-8)
X_cnn = X_norm[..., np.newaxis]   # Add channel dim → (N, 128, 130, 1)

# Save global stats (for inference-time normalization)
global_mean = float(X.mean())
global_std = float(X.std())

X_train, X_test, y_train, y_test = train_test_split(
    X_cnn, y_cat, test_size=0.15, random_state=42, stratify=y_enc
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

# 2D-CNN architecture
model = Sequential([
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=X_train.shape[1:]),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.2),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.2),

    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.3),

    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.4),

    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-6, verbose=1)
]

print('\n🚀 Training on GPU...\n')
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=80,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'\n📈 Test Accuracy: {acc*100:.2f}%   |   Loss: {loss:.4f}')

## Cell 5 — Save & Download Model Files

After this cell runs, your browser will download 3 files. Move them into:

```
m:\MENTOR-PRO\intelliview-python-engine\models\
```

In [ ]:
import os, json
from google.colab import files

os.makedirs('/content/models', exist_ok=True)

# Save the trained model
model.save('/content/models/speech_emotion_model.h5')

# Save normalization stats (used at inference time)
np.save('/content/models/global_mean.npy', np.array([global_mean]))
np.save('/content/models/global_std.npy', np.array([global_std]))

# Save the label class names
np.save('/content/models/label_classes.npy', le.classes_)

# Save the architecture metadata so the local analyzer knows input shape
meta = {
    'input_shape': list(X_train.shape[1:]),
    'n_mels': N_MELS,
    'n_frames': N_FRAMES,
    'sample_rate': SAMPLE_RATE,
    'duration': DURATION,
    'classes': list(le.classes_),
    'test_accuracy': float(acc),
    'model_type': '2d_cnn_mel_spectrogram'
}
with open('/content/models/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

model_size = os.path.getsize('/content/models/speech_emotion_model.h5') / (1024*1024)
print(f'\n💾 Model size: {model_size:.1f} MB')
print(f'📈 Test accuracy: {acc*100:.2f}%')
print(f'\n⬇️  Downloading 5 files to your computer...\n')

for fname in ['speech_emotion_model.h5', 'global_mean.npy', 'global_std.npy',
              'label_classes.npy', 'model_meta.json']:
    files.download(f'/content/models/{fname}')

print('\n✅ Done! Move the downloaded files into:')
print('   m:\\MENTOR-PRO\\intelliview-python-engine\\models\\')